# Week 3 — Task 1: Retrieval-Augmented Generation (RAG) over PDFs with ChromaDB

This notebook implements **Task 1 only** of the Week 3 assignment: a complete,
reusable RAG pipeline over a small collection of local PDFs.

**What this notebook does:**
1. Automatically discovers every PDF in the `data/` folder (no hardcoded filenames).
2. Extracts text from each PDF with `pypdf`.
3. Splits the text into configurable, overlapping chunks.
4. Generates embeddings for each chunk with `sentence-transformers`.
5. Stores chunks + embeddings + metadata (source file, page number) in `ChromaDB`.
6. Given a question, retrieves the top-k most relevant chunks and shows their
   source, page number, and similarity score.
7. Passes the retrieved context and the question to the **same OpenRouter
   client built in Week 2's `ChatApp.ipynb`** — no request logic is
   reimplemented here — and instructs the model to answer *only* from that
   context.

Tasks 2–5 are **not** implemented here; this notebook is meant to be a clean,
importable foundation for them.


## Reusing the Week 2 OpenRouter Client

Per the assignment, the existing `OpenRouterClient` / request logic from
`ChatApp.ipynb` is reused as-is — nothing about it is recreated or modified
here.

To do that from inside a notebook, we import `ChatApp.ipynb` **as a module**
using the `import_ipynb` package, instead of copy-pasting its code. This runs
`ChatApp.ipynb`'s cells once (defining its classes/functions) but — because
Python sets `__name__` to `"ChatApp"` rather than `"__main__"` during an
import — it does **not** trigger `ChatApp`'s interactive `main()` loop or
block waiting for terminal input.

> **Folder requirement:** this notebook must sit in the **same directory** as
> `ChatApp.ipynb`, `CONSTANTS.py`, and `.env` (all from Week 2), since the
> import below resolves `ChatApp` and its `CONSTANTS` import relative to the
> current working directory.


In [1]:
# --- Reuse the Week 2 OpenRouter client, unmodified ---
import import_ipynb          # lets us `import` a .ipynb file like a .py module
import ChatApp                # Week 2 notebook: OpenRouterClient, load_api_key, MODELS, etc.

print("Reused from ChatApp.ipynb:")
print(" - OpenRouterClient  :", ChatApp.OpenRouterClient)
print(" - OpenRouterAPIError:", ChatApp.OpenRouterAPIError)
print(" - load_api_key()    :", ChatApp.load_api_key)
print(" - MODELS            :", ChatApp.MODELS)

Reused from ChatApp.ipynb:
 - OpenRouterClient  : <class 'ChatApp.OpenRouterClient'>
 - OpenRouterAPIError: <class 'ChatApp.OpenRouterAPIError'>
 - load_api_key()    : <function load_api_key at 0x0000021D0949ADE0>
 - MODELS            : {'1': 'meta-llama/llama-3.1-8b-instruct', '2': 'google/gemma-3-27b-it', '3': 'nvidia/nemotron-nano-9b-v2:free'}


## Setup: Install & Import Packages

This notebook additionally needs:

| Package | Purpose |
|---|---|
| `pypdf` | Extract text from PDF files |
| `sentence-transformers` | Generate chunk/question embeddings |
| `chromadb` | Store and query embeddings (vector database) |
| `import-ipynb` | Import `ChatApp.ipynb` as a reusable module |

`requests` and `python-dotenv` are already required by `ChatApp.ipynb` and are
reused from there.


In [2]:
# Uncomment to install (safe to re-run; skips packages already installed)
# %pip install -q pypdf sentence-transformers chromadb import-ipynb

import os
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer

## The PDF Collection (`data/`)

Four small, publicly available educational PDFs are included in the `data/`
folder, covering Python, Machine Learning, Deep Learning, and Artificial
Intelligence — all commonly used, freely shared reference/cheat-sheet
materials. They're provided here **and** linked below for provenance.

| File | Topic | Author(s) | Source |
|---|---|---|---|
| `python_cheatsheet.pdf` | Python basics | Eric Matthes (*Python Crash Course*) | https://github.com/ehmatthes/pcc |
| `machine_learning_cheatsheet.pdf` | Machine Learning (Stanford CS229) | Afshine Amidi & Shervine Amidi | https://github.com/afshinea/stanford-cs-229-machine-learning |
| `deep_learning_cheatsheet.pdf` | Deep Learning (Stanford CS230) | Afshine Amidi & Shervine Amidi | https://github.com/afshinea/stanford-cs-230-deep-learning |
| `artificial_intelligence_cheatsheet.pdf` | Artificial Intelligence (Stanford CS221) | Afshine Amidi & Shervine Amidi | https://github.com/afshinea/stanford-cs-221-artificial-intelligence |

**No filenames are hardcoded anywhere below** — add, remove, or replace PDFs
in `data/` and the notebook will pick them up automatically on the next run.


## Configuration

All the "knobs" for this pipeline are collected in one place, per the
assignment's "configurable chunks/overlap" requirement. Change these and
re-run the notebook from here down to experiment.


In [3]:
# --------------------------------------------------------------------------- #
# Pipeline configuration
# --------------------------------------------------------------------------- #

DATA_DIR = Path("data")                 # Folder auto-scanned for PDFs
CHUNK_SIZE = 800                        # Characters per chunk
CHUNK_OVERLAP = 150                     # Characters of overlap between consecutive chunks
TOP_K = 4                               # Number of chunks retrieved per question

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # Small, fast sentence-transformers model

CHROMA_PERSIST_DIR = "chroma_store"     # ChromaDB writes its index here (on disk)
COLLECTION_NAME = "week3_pdf_knowledge_base"

GENERATION_MODEL_KEY = "1"              # Which ChatApp.MODELS entry answers questions
GENERATION_MODEL = ChatApp.MODELS[GENERATION_MODEL_KEY]

print(f"Chunking : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} char overlap")
print(f"Retrieval: top-{TOP_K} chunks")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Generation model (via ChatApp): {GENERATION_MODEL}")

Chunking : 800 chars, 150 char overlap
Retrieval: top-4 chunks
Embedding model: all-MiniLM-L6-v2
Generation model (via ChatApp): meta-llama/llama-3.1-8b-instruct


## Step 1 — Discover PDFs in `data/`

`discover_pdfs()` scans the data folder for `*.pdf` files at run time. There
is no list of expected filenames anywhere in this notebook — dropping a new
PDF into `data/` and re-running is all that's needed to include it.

**Error handling:** raises a clear `FileNotFoundError` if the folder is
missing or contains no PDFs, instead of failing later with a confusing error.


In [4]:
def discover_pdfs(data_dir: Path) -> List[Path]:
    """
    Find every PDF file inside `data_dir`.

    Args:
        data_dir: Folder to scan for PDF files.

    Returns:
        A sorted list of Paths to `.pdf` files.

    Raises:
        FileNotFoundError: If the folder doesn't exist or contains no PDFs.
    """
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Data directory '{data_dir}' does not exist. "
            f"Create it and add at least one PDF file."
        )

    pdf_paths = sorted(data_dir.glob("*.pdf"))

    if not pdf_paths:
        raise FileNotFoundError(
            f"No PDF files found in '{data_dir}'. Add at least one .pdf file."
        )

    return pdf_paths


pdf_paths = discover_pdfs(DATA_DIR)
print(f"Discovered {len(pdf_paths)} PDF(s) in '{DATA_DIR}/':")
for path in pdf_paths:
    print(f"  - {path.name}")

Discovered 4 PDF(s) in 'data/':
  - artificial_intelligence_cheatsheet.pdf
  - deep_learning_cheatsheet.pdf
  - machine_learning_cheatsheet.pdf
  - python_cheatsheet.pdf


## Step 2 — Extract Text with `pypdf`

`extract_pdf_pages()` reads a single PDF and returns one record per page that
contains extractable text, tagged with its page number and source filename
(the metadata the assignment asks us to keep).

**Error handling:**
- If a PDF can't be opened at all (corrupted file, wrong format, etc.), the
  file is skipped with a warning instead of crashing the whole run.
- If an individual page fails to extract, that page is skipped and the rest
  of the PDF is still processed.
- If a PDF yields no extractable text at all (e.g. a scanned image with no
  OCR layer), a warning is printed so it's obvious why that file contributed
  nothing to the knowledge base.


In [5]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    Extract text from every page of a PDF.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        A list of {"text", "page", "source"} dicts, one per page that had
        extractable text. Returns an empty list (with a printed warning)
        if the file can't be read or has no extractable text.
    """
    pages: List[Dict[str, Any]] = []

    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        print(f"  WARNING: could not open '{pdf_path.name}' — skipping. ({exc})")
        return pages

    for page_number, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception as exc:
            print(f"  WARNING: failed to extract page {page_number} of "
                  f"'{pdf_path.name}' — skipping that page. ({exc})")
            continue

        page_text = page_text.strip()
        if page_text:
            pages.append({
                "text": page_text,
                "page": page_number,
                "source": pdf_path.name,
            })

    if not pages:
        print(f"  WARNING: no extractable text found in '{pdf_path.name}' "
              f"(it may be a scanned/image-only PDF).")

    return pages

## Step 3 — Split Text into Configurable Chunks

`chunk_text()` slides a fixed-size window (`CHUNK_SIZE` characters) across a
page's text, moving forward by `CHUNK_SIZE - CHUNK_OVERLAP` characters each
time — so consecutive chunks share `CHUNK_OVERLAP` characters of context.
This keeps ideas that fall near a chunk boundary from being split without any
surrounding context in either chunk.

**Error handling:** raises a `ValueError` immediately if `CHUNK_OVERLAP` is
set greater than or equal to `CHUNK_SIZE`, which would otherwise cause the
sliding window to loop forever without advancing.


In [6]:
def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    """
    Split `text` into overlapping, fixed-size chunks.

    Args:
        text: The text to split.
        chunk_size: Maximum number of characters per chunk.
        chunk_overlap: Number of characters shared between consecutive chunks.

    Returns:
        A list of text chunks, in order.

    Raises:
        ValueError: If chunk_overlap >= chunk_size (would never advance).
    """
    if chunk_overlap >= chunk_size:
        raise ValueError(
            f"chunk_overlap ({chunk_overlap}) must be smaller than "
            f"chunk_size ({chunk_size})."
        )

    chunks: List[str] = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_length:
            break
        start = end - chunk_overlap  # step forward, keeping the overlap

    return chunks

## Step 4 — Build Chunk Records (Extraction + Chunking + Metadata)

`build_chunk_records()` ties Steps 2 and 3 together: for every discovered
PDF, extract each page's text, split it into chunks, and attach the metadata
this pipeline needs downstream — source filename, page number, and a
deterministic `id` (`<filename>_p<page>_c<chunk index>`) that ChromaDB will
use as the unique identifier for each entry.


In [7]:
def build_chunk_records(
    pdf_paths: List[Path],
    chunk_size: int,
    chunk_overlap: int,
) -> List[Dict[str, Any]]:
    """
    Extract and chunk every PDF, attaching source/page/id metadata.

    Args:
        pdf_paths: PDFs to process (from discover_pdfs()).
        chunk_size: Passed through to chunk_text().
        chunk_overlap: Passed through to chunk_text().

    Returns:
        A list of chunk records: {"id", "text", "source", "page", "chunk_index"}.
    """
    records: List[Dict[str, Any]] = []

    for pdf_path in pdf_paths:
        print(f"Processing '{pdf_path.name}'...")
        pages = extract_pdf_pages(pdf_path)

        for page_info in pages:
            page_chunks = chunk_text(page_info["text"], chunk_size, chunk_overlap)

            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "id": f"{pdf_path.stem}_p{page_info['page']}_c{chunk_index}",
                    "text": chunk,
                    "source": page_info["source"],
                    "page": page_info["page"],
                    "chunk_index": chunk_index,
                })

    return records


chunk_records = build_chunk_records(pdf_paths, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nBuilt {len(chunk_records)} chunks total.")
per_file_counts = {}
for record in chunk_records:
    per_file_counts[record["source"]] = per_file_counts.get(record["source"], 0) + 1
for source, count in per_file_counts.items():
    print(f"  - {source}: {count} chunks")

Processing 'artificial_intelligence_cheatsheet.pdf'...
Processing 'deep_learning_cheatsheet.pdf'...
Processing 'machine_learning_cheatsheet.pdf'...
Processing 'python_cheatsheet.pdf'...

Built 381 chunks total.
  - artificial_intelligence_cheatsheet.pdf: 82 chunks
  - deep_learning_cheatsheet.pdf: 54 chunks
  - machine_learning_cheatsheet.pdf: 70 chunks
  - python_cheatsheet.pdf: 175 chunks


## Step 5 — Generate Embeddings with `sentence-transformers`

`load_embedding_model()` loads a small, fast sentence-embedding model
(`all-MiniLM-L6-v2` by default), and `embed_texts()` encodes a batch of chunk
texts into vectors that capture their semantic meaning — the vectors that
make similarity search possible.

**Error handling:** both model loading and encoding are wrapped in
`try/except`, re-raised as a clear `RuntimeError` if the model fails to
download/load or if encoding fails for any reason (e.g. out of memory).


In [8]:
def load_embedding_model(model_name: str) -> SentenceTransformer:
    """
    Load a sentence-transformers embedding model.

    Args:
        model_name: Name of a sentence-transformers model on Hugging Face.

    Returns:
        The loaded SentenceTransformer model.

    Raises:
        RuntimeError: If the model fails to download or load.
    """
    try:
        return SentenceTransformer(model_name)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load embedding model '{model_name}'. Check your "
            f"internet connection (the model downloads from Hugging Face on "
            f"first use) and try again. Original error: {exc}"
        ) from exc


def embed_texts(model: SentenceTransformer, texts: List[str]) -> List[List[float]]:
    """
    Generate embeddings for a list of texts.

    Args:
        model: A loaded SentenceTransformer model.
        texts: The texts to embed.

    Returns:
        A list of embedding vectors (one per input text), as plain lists of
        floats (ChromaDB expects plain lists, not numpy arrays).

    Raises:
        RuntimeError: If embedding generation fails.
    """
    try:
        embeddings = model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
    except Exception as exc:
        raise RuntimeError(f"Embedding generation failed: {exc}") from exc

    return embeddings.tolist()


embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

chunk_texts = [record["text"] for record in chunk_records]
t0 = time.time()
chunk_embeddings = embed_texts(embedding_model, chunk_texts)
print(f"Generated {len(chunk_embeddings)} embeddings "
      f"(dimension={len(chunk_embeddings[0])}) in {time.time() - t0:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Generated 381 embeddings (dimension=384) in 9.09s


## Step 6 — Store Chunks + Embeddings + Metadata in ChromaDB

`get_chroma_collection()` opens (or creates) a persistent ChromaDB collection
on disk, configured for **cosine similarity**. `index_chunks()` then
`upsert`s every chunk's text, embedding, and metadata (source filename, page
number, chunk index) into it.

Using `upsert` with the deterministic IDs from Step 4 means re-running this
notebook on the same PDFs updates existing entries instead of duplicating
them.

**Error handling:** both collection setup and the upsert call are wrapped in
`try/except`, surfaced as a clear `RuntimeError` on failure (e.g. a corrupted
local index or a dimension mismatch between old and new embeddings).


In [9]:
def get_chroma_collection(persist_dir: str, collection_name: str):
    """
    Open (or create) a persistent ChromaDB collection using cosine similarity.

    Args:
        persist_dir: Local folder where ChromaDB stores its index.
        collection_name: Name of the collection to open/create.

    Returns:
        A ChromaDB Collection object.

    Raises:
        RuntimeError: If the ChromaDB client or collection can't be created.
    """
    try:
        chroma_client = chromadb.PersistentClient(path=persist_dir)
        collection = chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},  # smaller distance = more similar
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

    return collection


def index_chunks(collection, records: List[Dict[str, Any]], embeddings: List[List[float]]) -> None:
    """
    Upsert chunk text, embeddings, and metadata into a ChromaDB collection.

    Args:
        collection: A ChromaDB Collection (from get_chroma_collection()).
        records: Chunk records from build_chunk_records().
        embeddings: Embeddings from embed_texts(), aligned with `records`.

    Raises:
        RuntimeError: If the upsert fails.
    """
    try:
        collection.upsert(
            ids=[record["id"] for record in records],
            embeddings=embeddings,
            documents=[record["text"] for record in records],
            metadatas=[
                {
                    "source": record["source"],
                    "page": record["page"],
                    "chunk_index": record["chunk_index"],
                }
                for record in records
            ],
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to store chunks in ChromaDB: {exc}") from exc


collection = get_chroma_collection(CHROMA_PERSIST_DIR, COLLECTION_NAME)
index_chunks(collection, chunk_records, chunk_embeddings)
print(f"ChromaDB collection '{COLLECTION_NAME}' now holds {collection.count()} chunks.")

ChromaDB collection 'week3_pdf_knowledge_base' now holds 381 chunks.


## Step 7 — Retrieve the Top-K Most Relevant Chunks

`retrieve_relevant_chunks()` embeds the user's question with the same
embedding model used for the chunks, then queries ChromaDB for the `top_k`
nearest chunks. Since the collection is configured for cosine similarity,
ChromaDB's returned "distance" converts to a similarity score as
`similarity = 1 - distance` (closer to 1 = more relevant).

`display_retrieved_chunks()` prints each retrieved chunk's source file, page
number, and similarity score, plus a text preview — exactly what the
assignment asks to be shown before the answer is generated.


In [10]:
def retrieve_relevant_chunks(
    question: str,
    embedding_model: SentenceTransformer,
    collection,
    top_k: int,
) -> List[Dict[str, Any]]:
    """
    Retrieve the top_k chunks most semantically similar to `question`.

    Args:
        question: The user's question.
        embedding_model: The SentenceTransformer used to embed chunks.
        collection: The ChromaDB collection to query.
        top_k: Number of chunks to retrieve.

    Returns:
        A list of {"text", "source", "page", "similarity"} dicts, ordered
        from most to least relevant.

    Raises:
        RuntimeError: If embedding the question or querying ChromaDB fails.
    """
    try:
        question_embedding = embedding_model.encode(
            [question], convert_to_numpy=True
        ).tolist()
    except Exception as exc:
        raise RuntimeError(f"Failed to embed the question: {exc}") from exc

    try:
        results = collection.query(
            query_embeddings=question_embedding,
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB query failed: {exc}") from exc

    retrieved_chunks = []
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for doc_text, metadata, distance in zip(documents, metadatas, distances):
        retrieved_chunks.append({
            "text": doc_text,
            "source": metadata["source"],
            "page": metadata["page"],
            "similarity": 1 - distance,
        })

    return retrieved_chunks


def display_retrieved_chunks(retrieved_chunks: List[Dict[str, Any]]) -> None:
    """Print each retrieved chunk's source, page, similarity score, and a preview."""
    if not retrieved_chunks:
        print("No relevant chunks were found in the knowledge base.")
        return

    print(f"Retrieved {len(retrieved_chunks)} relevant chunk(s):\n")
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        preview = chunk["text"][:250] + ("..." if len(chunk["text"]) > 250 else "")
        print(f"[{rank}] source={chunk['source']}  page={chunk['page']}  "
              f"similarity={chunk['similarity']:.3f}")
        print(f"    {preview}\n")

## Step 8 — Generate a Grounded Answer (via the Reused OpenRouter Client)

`build_rag_messages()` assembles a chat message list: a system prompt that
instructs the model to answer **only** from the retrieved context (and to say
so plainly if the answer isn't there), followed by a user message containing
the retrieved chunks and the question.

`generate_answer()` then calls `client.send_chat(...)` — the exact same
method defined in `ChatApp.ipynb`'s `OpenRouterClient` — to get the model's
response. All of OpenRouter's error handling (rate limits, invalid key,
timeouts, etc.) comes for free from that reused client; here we just catch
`ChatApp.OpenRouterAPIError` and turn it into a readable message instead of
letting the notebook crash.


In [11]:
RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the "
    "context provided below. Do not use any outside knowledge, and do not "
    "guess. If the answer cannot be found in the provided context, respond "
    "exactly with: 'The information is not available in the provided "
    "context.' Always mention which source/page the answer came from when "
    "you do find it in the context."
)


def build_rag_messages(question: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """
    Build the chat messages for a retrieval-augmented question.

    Args:
        question: The user's question.
        retrieved_chunks: Chunks from retrieve_relevant_chunks().

    Returns:
        A list of {"role", "content"} messages ready for OpenRouterClient.send_chat().
    """
    context_blocks = [
        f"[Source: {chunk['source']}, page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ]
    context_text = "\n\n---\n\n".join(context_blocks)

    user_message = (
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer the question using only the context above."
    )

    return [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]


def generate_answer(
    client: "ChatApp.OpenRouterClient",
    model: str,
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
) -> "tuple[str, Optional[float]]":
    """
    Ask the (reused) OpenRouter client to answer a question from retrieved context.

    Args:
        client: An instance of ChatApp.OpenRouterClient.
        model: The OpenRouter model identifier to use for generation.
        question: The user's question.
        retrieved_chunks: Chunks from retrieve_relevant_chunks().

    Returns:
        A tuple of (answer_text, elapsed_seconds). elapsed_seconds is None if
        the call failed before a response was received.
    """
    messages = build_rag_messages(question, retrieved_chunks)

    try:
        result, elapsed = client.send_chat(model, messages)
    except ChatApp.OpenRouterAPIError as exc:
        return f"Could not generate an answer: {exc}", None

    try:
        answer = result["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return "Received an unexpected response format from the API.", None

    return answer, elapsed

## Step 9 — Put It All Together: `ask()`

`ask()` is the single entry point for this RAG pipeline: retrieve the
relevant chunks for a question, display them, pass them to the reused
OpenRouter client, and print the final grounded answer. Later tasks can build
on top of this one function.


In [12]:
def ask(question: str, top_k: int = TOP_K) -> str:
    """
    Answer a question using the RAG pipeline built above.

    Args:
        question: The user's question.
        top_k: Number of chunks to retrieve (defaults to the TOP_K setting).

    Returns:
        The model's answer text.
    """
    print(f'Question: "{question}"\n')

    retrieved_chunks = retrieve_relevant_chunks(question, embedding_model, collection, top_k)
    display_retrieved_chunks(retrieved_chunks)

    if not retrieved_chunks:
        answer = "The information is not available in the provided context."
        print(f"=== Answer ===\n{answer}")
        return answer

    answer, elapsed = generate_answer(client, GENERATION_MODEL, question, retrieved_chunks)

    print("=== Answer ===")
    if elapsed is not None:
        print(f"(generated by {GENERATION_MODEL} in {elapsed:.2f}s)\n")
    print(answer)

    return answer

## Load the OpenRouter API Key & Client (Reused from Week 2)

This calls `ChatApp.load_api_key()` and `ChatApp.OpenRouterClient(...)`
exactly as defined in Week 2 — nothing is redefined here. The only addition
is catching the `SystemExit` that `load_api_key()` raises when the key is
missing, so a missing `.env` file produces a clear notebook error instead of
an opaque kernel interruption.


In [13]:
try:
    api_key = ChatApp.load_api_key()
except SystemExit as exc:
    raise RuntimeError(
        "OPENROUTER_API_KEY is missing. Make sure a '.env' file (see "
        "ChatApp.ipynb's README from Week 2) is present in this same folder "
        "before continuing."
    ) from exc

client = ChatApp.OpenRouterClient(api_key)
print("OpenRouter client ready. Using model:", GENERATION_MODEL)

OpenRouter client ready. Using model: meta-llama/llama-3.1-8b-instruct


## Demo

Two example questions below:
1. One clearly answerable from the PDFs in `data/`.
2. One clearly **not** covered by them — to confirm the "not available in the
   provided context" behavior works as required.


In [14]:
_ = ask("What is gradient descent and how is it used in machine learning?")

Question: "What is gradient descent and how is it used in machine learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=2  similarity=0.541
    ) + (1 − y) log(1 − z)
]
Linear regression Logistic regression SVM Neural Network
ÌCost function– The cost functionJ is commonly used to assess the performance of a model,
and is deﬁned with the loss functionL as follows:
J(θ) =
m∑
i=1
L(hθ(x(i)),y (...

[2] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.527
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. The
vocabulary around neural networks architectures is described in the ﬁ...

[3] source=deep_learning_cheatsheet.pdf  page=13  similarity=0.511
    h, let alone a normal-sized training set.
Ì Gradient checking – Gradient checking is a method used during the implementation of
the backward pass of a neura

In [15]:
_ = ask("What is the capital of France?")  # not covered by these PDFs — should decline

Question: "What is the capital of France?"

Retrieved 4 relevant chunk(s):

[1] source=python_cheatsheet.pdf  page=1  similarity=0.158
    stored as a string. 
Prompting for a value 
name = input("What's your name? ") 
print("Hello, " + name + "!") 
Prompting for numerical input 
age = input("How old are you? ") 
age = int(age) 
 
pi = input("What's the value of pi? ") 
pi = float(pi)

[2] source=deep_learning_cheatsheet.pdf  page=10  similarity=0.158
    t′> = 1
Remark: the attention scores are commonly used in image captioning and machine translation.
Ì Attention weight – The amount of attention that the outputy<t> should pay to the
activationa<t′> is given byα<t,t′> computed as follows:
α<t,t′> = e...

[3] source=python_cheatsheet.pdf  page=16  similarity=0.149
    le(self): 
        """Test names like David Lee Roth.""" 
        full_name = get_full_name('david', 
                'roth', 'lee') 
        self.assertEqual(full_name, 
                'David Lee Roth') 
 
unittest.main

## Try Your Own Question

Run the cell below to type a question interactively.


In [16]:
user_question = input("Enter your question: ").strip()
if user_question:
    _ = ask(user_question)
else:
    print("No question entered.")

Question: "What are the types of Machine Learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.501
    CS 229 – Machine Learning https://stanford.edu/~shervine
Super VIP Cheatsheet: Machine Learning
Afshine Amidi and Shervine Amidi
October 6, 2018
Contents
1 Supervised Learning 2
1.1 Introduction to Supervised Learning. . . . . . . . . . . . . . . . ....

[2] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.445
    t Vector Machines . . . . . . . . . . . . . . . . . . . . . . . . 3
1.5 Generative Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
1.5.1 Gaussian Discriminant Analysis . . . . . . . . . . . . . . . . . 4
1.5.2 Naive Bayes . . . . . ....

[3] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.414
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. Th

## Summary

This notebook built a complete, reusable RAG pipeline:

`discover_pdfs` → `extract_pdf_pages` → `chunk_text` → `build_chunk_records`
→ `embed_texts` → `index_chunks` (ChromaDB) → `retrieve_relevant_chunks` →
`generate_answer` (via the reused `ChatApp.OpenRouterClient`) → `ask()`

All of it is driven by the configuration block near the top (`CHUNK_SIZE`,
`CHUNK_OVERLAP`, `TOP_K`, `EMBEDDING_MODEL_NAME`, etc.) and by whatever PDFs
happen to be in `data/` at run time — no filenames are hardcoded anywhere.

**Not implemented here (by design):** Tasks 2, 3, 4, and 5. This notebook is
meant to be the foundation those tasks build on top of.
